In [46]:
import pandas as pd

curriculo = pd.read_excel('Currículo.xlsx')



display(curriculo)

C:\Users\pjcla\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\openpyxl\worksheet\_reader.py:223: UserWarning: Cell C446 is marked as a date but the serial value 6705391.0 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)


,Coluna,Nº processo,Data,Sexo,Idade,Área,Diagnóstico,Origem,Ambulatório,Tipo de Cirurgia,Cirurgião,Procedimento,Anatomia Patológica,Clavien-Dindo,Descrição Complicação,Consulta,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,1,2012163,2025-01-08,M,67,Pele e Tecidos Moles,Lipoma,Programado,Sim,Peq. Cirurgia,Ajudante,Excisão de lipoma,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,110561,2025-01-08,M,65,Pele e Tecidos Moles,Lipoma,Programado,Sim,Peq. Cirurgia,Ajudante,Excisão de lipoma,NaN,NaN,NaN,NaN,NaN,Pequena Cirurgia,Principal,Ajudante
2,3,80999,2025-01-08,F,43,Pele e Tecidos Moles,Verruga,Programado,Sim,Peq. Cirurgia,Ajudante,Excisão de verruga,NaN,NaN,NaN,NaN,NaN,Quisto sebáceo,96,11
3,4,149552,2025-01-08,F,48,Pele e Tecidos Moles,Quisto sebáceo,Programado,Sim,Peq. Cirurgia,Ajudante,Excisão de quisto sebáceo,NaN,NaN,NaN,NaN,NaN,Lipoma,39,8
4,5,354032,2025-01-13,M,87,Cirurgia dos Membros,Isquémia irreversível do membro,Urgência,Não,Aberta,Ajudante,Amputação supracondiliana,NaN,NaN,NaN,NaN,NaN,Fibromas pêndulos,48,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
606,600,189998,2025-12-30,F,58,Pele e Tecidos Moles,Lipoma,NaN,Sim,Peq. Cirurgia,Principal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
607,601,371226,2025-12-30,M,45,Pele e Tecidos Moles,Quisto sebáceo,NaN,Sim,Peq. Cirurgia,Principal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608,602,21601,2025-12-30,F,55,Pele e Tecidos Moles,Quisto sebáceo,NaN,Sim,Peq. Cirurgia,Principal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
609,603,2110100,2025-12-30,M,26,Pele e Tecidos Moles,Quisto sebáceo,NaN,Sim,Peq. Cirurgia,Principal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# read data from mysql
import mysql.connector
from mysql.connector import Error
import pandas as pd
import os

def read_data_from_mysql(query, db_config):
    try:
        # Connect to the MySQL database
        connection = mysql.connector.connect(**db_config)
        if connection.is_connected():
            # Execute the query and fetch the data into a DataFrame
            df = pd.read_sql(query, connection)
            return df
    except Error as e:
        print(f"Error: {e}")
    finally:
        if connection.is_connected():
            connection.close()
            print("MySQL connection is closed")
            

# Define the database configuration
db_config = {
    'host': 'localhost',
    'user':'root',
    'password':'',
    'database':'medtracker'
}

def ler_tabela(tabela):
    query = f"SELECT * FROM {tabela}"
    # Read data from MySQL and store it in a DataFrame
    db = read_data_from_mysql(query, db_config)
    display(db)

def listar_colunas(tabela):
    query = f"SHOW COLUMNS FROM {tabela}"
    # Read data from MySQL and store it in a DataFrame
    db = read_data_from_mysql(query, db_config)
    display(db)
    
# find
def find(tabela, coluna, valor):
    query = f"SELECT * FROM {tabela} WHERE {coluna} = '{valor}'"
    # Read data from MySQL and store it in a DataFrame
    db = read_data_from_mysql(query, db_config)
    if db.empty:
        return True
    else:        
        return False
    

In [ ]:
import mysql.connector
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='',
    database='medtracker'
)
cursor = conn.cursor()

query  ="Insert into utentes (processo, idade, sexo) values (%s, %s, %s)"
for index, utente in curriculo.iterrows():
    processo = str(utente['Nº processo'])
    idade = utente['Idade']
    sexo = utente['Sexo']

    if find('utentes', 'processo', processo):
        print('Utente não encontrado')
        cursor.execute(query, (processo, idade, sexo))
        conn.commit()

In [ ]:
# zona_anatomicas
query  ="Insert into zona_anatomicas (nome, user_id) values (%s, %s)"
for index, zona in curriculo.iterrows():
    if find('zona_anatomicas', 'nome',  zona['Área']):
        cursor.execute(query, (zona['Área'], 1))
        conn.commit()

In [48]:
#diagnosticos
query  ="Insert into diagnosticos (nome, user_id, zona_anatomica_id) values (%s, %s, %s)"
for index, diagnostico in curriculo.iterrows():
    if find('diagnosticos', 'nome',  diagnostico['Diagnóstico']):
        cursor.execute(f"SELECT id FROM zona_anatomicas WHERE nome = '{diagnostico['Área']}'")
        zona_anatomica_id = cursor.fetchone()[0]
        cursor.execute(query, (diagnostico['Diagnóstico'], 1, zona_anatomica_id))
        conn.commit()   

MySQL connection is closed
MySQL connection is closed
MySQL connection is closed


C:\Users\pjcla\AppData\Local\Temp\ipykernel_27360\3820529789.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


ProgrammingError: 1054 (42S22): Unknown column 'zona_anatomica_id' in 'field list'